In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mga karaniwang ginagamit na parameter para sa transpilation

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Inilalarawan ng pahinang ito ang ilan sa mga mas karaniwang ginagamit na parameter para sa lokal na transpilation. Ang mga parameter na ito ay kino-configure gamit ang mga argumento sa [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) o [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Antas ng pagtatantya
Maaari mong gamitin ang approximation degree upang tukuyin kung gaano kalapit mo gustong itugma ang resultang circuit sa nais (input) na circuit. Ito ay isang float sa saklaw na (0.0 - 1.0), kung saan ang 0.0 ay maximum na pagtatantya at ang 1.0 (default) ay walang pagtatantya. Ang mas maliit na mga halaga ay nagpapalitan ng katumpakan ng output para sa kadalian ng pagpapatupad (ibig sabihin, mas kaunting mga gate). Ang default na halaga ay 1.0.

Sa two-qubit unitary synthesis (ginagamit sa mga paunang yugto ng lahat ng antas at para sa yugto ng optimization na may optimization level 3), ang halagang ito ay tumutukoy sa target na fidelity ng output decomposition. Ibig sabihin, kung gaano karaming error ang ipinakilala kapag ang matrix representation ng isang circuit ay na-convert sa mga discrete gate. Kung ang approximation degree ay mas mababang halaga (mas maraming pagtatantya), ang output circuit mula sa synthesis ay mas mag-iiba mula sa input matrix, ngunit malamang na magkakaroon din ng mas kaunting mga gate (dahil ang anumang arbitrary na two-qubit na operasyon ay maaaring ma-decompose nang perpekto nang may pinakamataas na tatlong CX gate) at mas madaling patakbuhin.

Kapag ang approximation degree ay mas mababa sa 1.0, ang mga circuit na may isa o dalawang CX gate ay maaaring synthesized, na humahantong sa mas kaunting error mula sa hardware, ngunit mas marami mula sa pagtatantya. Dahil ang CX ang pinakamahirap na gate sa mga tuntunin ng error, maaaring kapaki-pakinabang na bawasan ang bilang ng mga ito sa halaga ng fidelity sa synthesis (ang pamamaraang ito ay ginamit upang mapataas ang quantum volume sa IBM&reg; devices: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Bilang halimbawa, bumubuo tayo ng random na two-qubit na `UnitaryGate` na isi-synthesize sa paunang yugto. Ang pagtatakda ng `approximation_degree` na mas mababa sa 1.0 ay maaaring bumuo ng isang approximate na circuit. Dapat din nating tukuyin ang `basis_gates` upang malaman ng paraan ng synthesis kung aling mga gate ang maaari nitong gamitin para sa approximate synthesis.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Nagbubunga ito ng output na `2` dahil ang pagtatantya ay nangangailangan ng mas kaunting CX gate.

<span id="seed"></span>
## Seed ng random number generator
Ang ilang bahagi ng transpiler ay stochastic, kaya ang paulit-ulit na mga run ng transpilation ay maaaring magbalik ng iba't ibang resulta. Upang makakuha ng reproducible na resulta, maaari mong itakda ang seed para sa pseudorandom number generator gamit ang argumento na `seed_transpiler`. Ang paulit-ulit na mga run gamit ang parehong seed ay magbabalik ng parehong mga resulta.

Halimbawa:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Paunang layout
Bago ang transpilation, ang mga qubit na nasa iyong circuit ay mga virtual qubit na hindi kinakailangang katumbas ng mga pisikal na qubit sa target na backend. Maaari mong tukuyin ang paunang pagmamapa ng mga virtual qubit sa mga pisikal na qubit gamit ang argumento na `initial_layout`. Tandaan na ang panghuling layout ng qubit ay maaaring mag-iba mula sa paunang layout dahil maaaring mag-permute ng mga qubit ang transpiler gamit ang mga swap gate o iba pang paraan.

Sa halimbawa sa ibaba, bumubuo tayo ng paunang layout para sa [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) mock backend sa pamamagitan ng paglikha ng isang [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout) na object. Ang aming layout ay nagmamapa ng unang qubit ng aming circuit sa qubit 5 ng Sherbrooke, at nagmamapa ng pangalawang qubit ng aming circuit sa qubit 6 ng Sherbrooke. Tandaan na ang mga pisikal na qubit ay palaging kinakatawan ng mga integer.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Bukod sa pagtukoy ng isang Layout object, maaari ka ring magpasa ng listahan ng mga integer, kung saan ang $i$-th na elemento ng listahan ay naglalaman ng pisikal na qubit na dapat imapa ang $i$-th na qubit. Halimbawa:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Maaari mong gamitin ang function na [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) upang bumuo ng diagram ng device graph na may impormasyon sa error at may mga label na pisikal na qubit. Maaari ka ring tumingin ng mga katulad na diagram sa pahina ng [Compute resources](https://quantum.cloud.ibm.com/computers).